# Home Credit Default Risk — Data Preprocessing

## Objective

This notebook prepares the Home Credit application data for machine
learning.

The preprocessing steps will include:

- Handling missing values
- Cleaning special values
- Encoding categorical variables
- Creating a training and validation split
- Preparing the data for the baseline model

In [12]:
import pandas as pd
import numpy as np

train = pd.read_csv('../data/application_train.csv')
test = pd.read_csv('../data/application_test.csv')

print("Train shape:", train.shape)
print("Test shape:", test.shape)

missing = train.isnull().sum().sort_values(ascending=False)

missing_percent = (
    train.isnull().mean() * 100
).sort_values(ascending=False)

missing_summary = pd.DataFrame({
    "Missing Count": missing,
    "Missing Percent": missing_percent
})

display(missing_summary.head(20))

missing_over_60 = missing_percent[missing_percent > 60]

print("Features with more than 60% missing:")
print(len(missing_over_60))

display(missing_over_60)

Train shape: (307511, 122)
Test shape: (48744, 121)


,Missing Count,Missing Percent
COMMONAREA_MEDI,214865,69.872297
COMMONAREA_AVG,214865,69.872297
COMMONAREA_MODE,214865,69.872297
NONLIVINGAPARTMENTS_MODE,213514,69.432963
NONLIVINGAPARTMENTS_AVG,213514,69.432963
NONLIVINGAPARTMENTS_MEDI,213514,69.432963
FONDKAPREMONT_MODE,210295,68.386172
LIVINGAPARTMENTS_MODE,210199,68.354953
LIVINGAPARTMENTS_AVG,210199,68.354953
LIVINGAPARTMENTS_MEDI,210199,68.354953


Features with more than 60% missing:
17


COMMONAREA_MEDI             69.872297
COMMONAREA_AVG              69.872297
COMMONAREA_MODE             69.872297
NONLIVINGAPARTMENTS_MODE    69.432963
NONLIVINGAPARTMENTS_AVG     69.432963
NONLIVINGAPARTMENTS_MEDI    69.432963
FONDKAPREMONT_MODE          68.386172
LIVINGAPARTMENTS_MODE       68.354953
LIVINGAPARTMENTS_AVG        68.354953
LIVINGAPARTMENTS_MEDI       68.354953
FLOORSMIN_AVG               67.848630
FLOORSMIN_MODE              67.848630
FLOORSMIN_MEDI              67.848630
YEARS_BUILD_MEDI            66.497784
YEARS_BUILD_MODE            66.497784
YEARS_BUILD_AVG             66.497784
OWN_CAR_AGE                 65.990810
dtype: float64

## Missing Value Strategy

The application dataset contains substantial missingness in several
features. Some variables are missing for more than 60% of applicants.

For the baseline preprocessing pipeline, features with more than 60%
missing values will be removed. This provides a simple and consistent
approach while avoiding heavy imputation of variables with very limited
observed data.

Features with lower levels of missingness will be retained and handled
through imputation during preprocessing.

This threshold is a modeling decision rather than a statement that the
underlying variables are unimportant. Later model iterations may
investigate whether alternative missing-value strategies improve
performance.

## Removing High-Missingness Features

17 features have more than 60% of their values missing.

For the baseline preprocessing pipeline, these features will be removed
because the majority of applicants do not have observed values for these
variables.

This reduces the amount of information that needs to be imputed and keeps
the baseline preprocessing approach relatively simple.

In [13]:
high_missing_features = missing_percent[missing_percent > 60].index

train = train.drop(columns=high_missing_features)
test = test.drop(columns=high_missing_features)

print("Removed features:", len(high_missing_features))
print("New train shape:", train.shape)
print("New test shape:", test.shape)


remaining_missing = train.isnull().sum().sort_values(ascending=False)

remaining_missing_percent = (
    train.isnull().mean() * 100
).sort_values(ascending=False)

remaining_missing_summary = pd.DataFrame({
    "Missing Count": remaining_missing,
    "Missing Percent": remaining_missing_percent
})

display(remaining_missing_summary.head(20))

Removed features: 17
New train shape: (307511, 105)
New test shape: (48744, 104)


,Missing Count,Missing Percent
LANDAREA_AVG,182590,59.376738
LANDAREA_MODE,182590,59.376738
LANDAREA_MEDI,182590,59.376738
BASEMENTAREA_AVG,179943,58.515956
BASEMENTAREA_MODE,179943,58.515956
BASEMENTAREA_MEDI,179943,58.515956
EXT_SOURCE_1,173378,56.381073
NONLIVINGAREA_AVG,169682,55.179164
NONLIVINGAREA_MEDI,169682,55.179164
NONLIVINGAREA_MODE,169682,55.179164


In [14]:
numeric_features = train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

print("\nCategorical columns:")
print(categorical_features)

Numerical features: 90
Categorical features: 15

Categorical columns:
['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE']


/var/folders/y1/p0h0n89x6c530xhk72bbnww40000gn/T/ipykernel_55775/3370895416.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = train.select_dtypes(


In [15]:
# Separate features from target
X = train.drop(columns=["TARGET"])
y = train["TARGET"]

# Identify numerical and categorical features
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

print("\nCategorical columns:")
print(categorical_features)

Numerical features: 89
Categorical features: 15

Categorical columns:
['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE']


/var/folders/y1/p0h0n89x6c530xhk72bbnww40000gn/T/ipykernel_55775/3739216715.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(
